# Lumina — Specialty Psych Worker (Layer B)

Runs **one** 8B psychology LLM on a free Colab **T4 GPU** and exposes it to your
Lumina backend over a public **ngrok** URL.

## Contract (must match `backend/services/psych_specialty.py`)

```
POST {base_url}/infer
    body:    {"prompt": str, "max_new_tokens": int = 256, "temperature": float = 0.7}
    returns: {"text": str}

GET  {base_url}/health
    returns: {"ok": true, "model": "<id>"}
```

## How to use

1. Runtime → **Change runtime type** → Hardware accelerator = **T4 GPU**.
2. In **Cell 1**, pick the `MODEL_ID` for the page you are powering.
3. (First run only) put your free **ngrok authtoken** in Cell 1.
4. **Runtime → Run all.** Wait ~3–5 min for the model to load.
5. Copy the printed `https://....ngrok-free.app` URL.
6. Paste it into the matching `COLAB_URL_*` in `backend/.env`, restart the
   backend. Done.

| Page        | MODEL_ID                                  | backend/.env var          |
|-------------|-------------------------------------------|---------------------------|
| /cbt        | `GMLHUHE/PsyLLM-8B`                        | `COLAB_URL_PSYLLM`        |
| /chat       | `Psychotherapy-LLM/PsyCoPref-Llama3-8B`   | `COLAB_URL_PSYCHOCOUNSEL` |
| /screening  | `aminabbasi/PsychoLexLLaMA-8B`            | `COLAB_URL_PSYCHOLEX`    |
| /dashboard  | `klyang/MentaLLaMA-chat-7B`               | `COLAB_URL_MENTALLAMA`   |

> The `/chat` repo `Psychotherapy-LLM/PsychoCounsel-Llama3-8B` was renamed to
> `PsyCoPref-Llama3-8B` (same weights, public, not gated) — use the new id above.

> One T4 holds **one** 8B model in 4-bit. To run all four, open this notebook
> in **four separate Colab tabs** (free tier allows it), each with a different
> `MODEL_ID`, and paste the four URLs. Layer B is **fail-soft**: any worker that
> is asleep or absent just falls back to the small HF classifier — the app
> never hard-fails.

## Cell 1 — Config (edit these two lines)

In [ ]:
# === EDIT ME ===========================================================
# Pick the model for the page you are powering (see table above).
MODEL_ID = "GMLHUHE/PsyLLM-8B"

# Free ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = ""

# Optional: HF token, only needed if MODEL_ID is a gated/private repo.
HF_TOKEN = ""
# =======================================================================

# Load model in 4-bit so an 8B fits in the T4's 16 GB VRAM. Set False only for
# <=3B models or an A100 runtime.
LOAD_IN_4BIT = True
PORT = 8000

assert NGROK_AUTHTOKEN, "Set NGROK_AUTHTOKEN in Cell 1 first."
print("MODEL_ID =", MODEL_ID)

## Cell 2 — Install deps (~1 min)

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes fastapi "uvicorn[standard]" pyngrok nest_asyncio

## Cell 3 — Load the model onto the T4 (~3–5 min)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), (
    "No GPU. Runtime → Change runtime type → T4 GPU, then Run all again."
)
print("GPU:", torch.cuda.get_device_name(0))

_hf = HF_TOKEN or None
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=_hf)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

quant = (
    BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    if LOAD_IN_4BIT
    else None
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=quant,
    token=_hf,
)
model.eval()
print("Loaded", MODEL_ID)

## Cell 4 — Inference fn (uses chat template when available)

In [ ]:
@torch.inference_mode()
def generate(prompt: str, max_new_tokens: int = 256, temperature: float = 0.7) -> str:
    # Prefer the model's own chat template (Llama-3 style) when present, so the
    # instruction-tuned weights behave as intended.
    if getattr(tokenizer, "chat_template", None):
        inputs = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            add_generation_prompt=True,
            return_tensors="pt",
        ).to(model.device)
        input_len = inputs.shape[1]
    else:
        enc = tokenizer(prompt, return_tensors="pt").to(model.device)
        inputs = enc.input_ids
        input_len = inputs.shape[1]

    do_sample = temperature and temperature > 0
    out = model.generate(
        inputs,
        max_new_tokens=int(max_new_tokens),
        do_sample=do_sample,
        temperature=float(temperature) if do_sample else None,
        top_p=0.9 if do_sample else None,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.1,
    )
    # Return only the newly generated tokens.
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()


print(generate("Say 'worker online' in three words.", max_new_tokens=16))

## Cell 5 — FastAPI app + ngrok tunnel → prints your public URL

In [ ]:
import nest_asyncio, uvicorn, threading
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok, conf

nest_asyncio.apply()

app = FastAPI(title="Lumina Specialty Psych Worker")


class InferIn(BaseModel):
    prompt: str
    max_new_tokens: int = 256
    temperature: float = 0.7


@app.get("/health")
def health():
    return {"ok": True, "model": MODEL_ID}


@app.post("/infer")
def infer(body: InferIn):
    text = generate(body.prompt, body.max_new_tokens, body.temperature)
    return {"text": text}


# Open the tunnel BEFORE the (blocking) server starts.
conf.get_default().auth_token = NGROK_AUTHTOKEN
ngrok.kill()  # clear any stale tunnel from a previous run
public_url = ngrok.connect(PORT, "http").public_url
print("\n" + "=" * 64)
print("  PUBLIC URL  ->", public_url)
print("  Paste this into backend/.env for this model's COLAB_URL_* var")
print("=" * 64 + "\n")

# Run uvicorn in a background thread so the cell stays responsive and the
# notebook does not die. Keep this tab open to keep the worker alive.
def _serve():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")


threading.Thread(target=_serve, daemon=True).start()
print("Worker serving. Leave this tab open.")

## Cell 6 (optional) — Smoke test the live endpoint

In [ ]:
import requests, time
time.sleep(2)
print("health:", requests.get(public_url + "/health", timeout=10).json())
r = requests.post(
    public_url + "/infer",
    json={"prompt": "In one sentence, what is cognitive reframing?", "max_new_tokens": 60},
    timeout=60,
)
print("infer:", r.json())